## Min identity 0.8 - low high input comparison in monoclonal antibody 1
### filtering peptides >= 7 and minimum 0.9 confidence score


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
from pathlib import Path
from Bio import SeqIO
import json
import os
import logging
import numpy as np
from scipy.stats import mannwhitneyu

# Import all our custom pipeline modules
from instanexus import preprocessing
from instanexus import assembly
from instanexus import clustering
from instanexus import alignment
from instanexus import consensus
from instanexus import visualization
from instanexus import helpers

# Set up logging to see the pipeline's progress
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
os.chdir('../../../../')

print(f"Current working directory: {os.getcwd()}")

In [ ]:
FIGURES_DIR = Path("figures")
print(FIGURES_DIR)

In [ ]:
# Path to the new raw data you want to test
#INPUT_CSV = "_archive/csv/v2/bsa.csv"
#INPUT_CSV = "inputs/bsa.csv"
INPUT_CSV_1 = "inputs/ma1_low.csv"
INPUT_CSV_2 = "inputs/ma1_high.csv"

BASE_OUTPUT_FOLDER = "outputs_notebook"

METADATA_PATH = "json/sample_metadata.json"
CONTAMINANTS_PATH = "fasta/contaminants.fasta"

RUN_NAME = Path(INPUT_CSV_1).stem
REFERENCE_MODE = True
CHAIN = "heavy"  # "heavy" or "light"

# Filtering params
CONFIDENCE_THRESHOLD = 0.8
MASS_ERR_LIMIT = 20
MIN_LENGTH = 7
MAX_IRT_ERROR = 60
MIN_ENTROPY = 1
PROSIT_FILTER = True
FDR_THRESHOLD = 0.1
Z_SCORE_THRESHOLD = -0.5

# Assembly params
ASSEMBLY_MODE = "greedy"
KMER_SIZE = 6
MIN_OVERLAP = 3
SIZE_THRESHOLD = 10

MIN_IDENTITY = 0.8
MAX_MISMATCHES = 0

# Clustering params
MIN_SEQ_ID = 0.85
COVERAGE = 0.8

In [ ]:
sample_metadata = preprocessing.get_sample_metadata(
    run="ma1", 
    chain=CHAIN, 
    json_path=METADATA_PATH
)

In [ ]:
proteases = sample_metadata["proteases"]
protein = sample_metadata["protein"]
protein_norm = preprocessing.normalize_sequence(protein)

In [ ]:
print(f"Sample uses proteases: {proteases}")
print(f"Protein sequence length: {len(protein)} amino acids")
print(f"Normalized protein sequence: {protein_norm}")

In [ ]:
ma1_low = pd.read_csv(INPUT_CSV_1)
ma1_high = pd.read_csv(INPUT_CSV_2)

In [ ]:
ma1_low.columns

In [ ]:
ma1_low["protease"] = ma1_low["experiment_name"].apply(
    lambda name: preprocessing.extract_protease(name, proteases)
)

protease_col = ma1_low.pop("protease")
ma1_low.insert(ma1_low.columns.get_loc("preds") + 1, "protease", protease_col)
ma1_low.head(3)

In [ ]:
ma1_low["protease"] = ma1_low["experiment_name"].apply(
    lambda name: preprocessing.extract_protease(name, proteases)
)

protease_col = ma1_low.pop("protease")
ma1_low.insert(ma1_low.columns.get_loc("preds") + 1, "protease", protease_col)
ma1_low.head(3)

In [ ]:
ma1_high["protease"] = ma1_high["experiment_name"].apply(
    lambda name: preprocessing.extract_protease(name, proteases)
)

protease_col = ma1_high.pop("protease")
ma1_high.insert(ma1_high.columns.get_loc("preds") + 1, "protease", protease_col)
ma1_high.head(3)

In [ ]:
print(ma1_low.shape[0])
print(ma1_high.shape[0])


In [ ]:
ma1_low = ma1_low.dropna(subset=["preds"])
ma1_high = ma1_high.dropna(subset=["preds"])

In [ ]:
print(ma1_low.shape[0])
print(ma1_high.shape[0])


In [ ]:
ma1_low["cleaned_preds"] = ma1_low["preds"].apply(preprocessing.remove_modifications)
ma1_high["cleaned_preds"] = ma1_high["preds"].apply(preprocessing.remove_modifications)

In [ ]:
# make a column called number_amino_acids
ma1_low["number_amino_acids"] = ma1_low["cleaned_preds"].apply(len)
ma1_high["number_amino_acids"] = ma1_high["cleaned_preds"].apply(len)

In [ ]:
# convert the column log_probs in the exponential scale
ma1_low["exp_log_probs"] = ma1_low["log_probs"].apply(np.exp)
ma1_high["exp_log_probs"] = ma1_high["log_probs"].apply(np.exp)

In [ ]:
# filter based on exp_log_probs and number_amino_acids: exp_log_probs > 0.9 and number_amino_acids >= 7
ma1_low_filtered = ma1_low[(ma1_low["exp_log_probs"] > 0.9) & (ma1_low["number_amino_acids"] >= 7)]
ma1_high_filtered = ma1_high[(ma1_high["exp_log_probs"] > 0.9) & (ma1_high["number_amino_acids"] >= 7)]

In [ ]:
print(ma1_low_filtered.shape[0])
print(ma1_high_filtered.shape[0])


In [ ]:
cleaned_psms_low = ma1_low_filtered["cleaned_preds"].tolist()
cleaned_psms_high = ma1_high_filtered["cleaned_preds"].tolist()

In [ ]:
print(MAX_MISMATCHES)
print(MIN_IDENTITY)

In [ ]:
mapped_psms_low = visualization.process_protein_contigs_scaffold(
    cleaned_psms_low, protein_norm, MAX_MISMATCHES, MIN_IDENTITY
)

mapped_psms_high = visualization.process_protein_contigs_scaffold(
    cleaned_psms_high, protein_norm, MAX_MISMATCHES, MIN_IDENTITY
)

In [ ]:
print(len(mapped_psms_low))
print(len(mapped_psms_high))

In [ ]:
df_psms_mapped_low = visualization.create_dataframe_from_mapped_sequences(data=mapped_psms_low)
df_psms_mapped_high = visualization.create_dataframe_from_mapped_sequences(data=mapped_psms_high)

In [ ]:
df_psms_mapped_low.head(3)

In [ ]:
stats_low = helpers.compute_assembly_statistics(
    df_psms_mapped_low, f"psms_{CHAIN}_low", "./outputs/_low_high_input", protein_norm
)

stats_high = helpers.compute_assembly_statistics(
    df_psms_mapped_high, f"psms_{CHAIN}_high", "./outputs/_low_high_input", protein_norm
)

## Barplot comparison

In [ ]:
base_folder = "./outputs/_low_high_input" # o il percorso corretto dove hai salvato i json

files_map = {
    ('Light Chain', 'Low Input'): os.path.join(base_folder, "psms_light_low_stats.json"),
    ('Light Chain', 'High Input'): os.path.join(base_folder, "psms_light_high_stats.json"),
    ('Heavy Chain', 'Low Input'): os.path.join(base_folder, "psms_heavy_low_stats.json"),
    ('Heavy Chain', 'High Input'): os.path.join(base_folder, "psms_heavy_high_stats.json"),
}

In [ ]:
data = []

for (chain, input_type), file_path in files_map.items():
    try:
        with open(file_path, 'r') as f:
            stats = json.load(f)
            coverage_pct = stats['coverage'] * 100 
            data.append({
                'Chain': chain,
                'Input': input_type,
                'Coverage': coverage_pct
            })
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        data.append({'Chain': chain, 'Input': input_type, 'Coverage': 0})

In [ ]:
df_plot = pd.DataFrame(data)

sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 6))

custom_palette = {'Low Input': '#e74c3c', 'High Input': '#2ecc71'}

ax = sns.barplot(
    data=df_plot,
    x='Chain',
    y='Coverage',
    hue='Input',
    palette=custom_palette,
    edgecolor="black",
    linewidth=1
)

plt.ylabel('Sequence Coverage (%)', fontsize=12)
plt.xlabel('')
plt.ylim(0, 110)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%',
    padding=3, fontsize=11, fontweight='bold')

plt.legend(title='Input Amount', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig(os.path.join(base_folder, "coverage_barplot.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(base_folder, "coverage_barplot.svg"), format='svg', bbox_inches='tight')
plt.show()